# FABLE Pyculator 2021 FreshForge Build Plan

This notebook shows how FABLE Pyculator, FreshForge, and Modelwright fit together when rebuilding a 2021 FABLE-C generated Python model from the 2021 source workbook.

FreshForge planning is not execution. FreshForge validates and plans the workflow graph; Modelwright still performs contract inference, code generation, model execution, and validation.

## Local artifacts

Expected local source workbook:

- `tmp/private-workbooks/2021_Open_FABLECalculator.xlsx`

Ignored generated-model working directory:

- `tmp/generated-models/fable-2021/`

This notebook writes only local build artifacts under `tmp/`. Do not commit the source workbook, generated JSON files, decompressed generated model, logs, or validation reports.

In [ ]:
from __future__ import annotations

import shlex
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

from fable_pyculator import (
    build_2021_notebook_spec,
    build_cached_workbook_validation_scenario,
    build_modelwright_freshforge_workflow,
    derive_output_refs,
    freshforge_2021_build_paths,
    write_freshforge_workflow,
    write_output_refs,
    write_validation_scenario,
)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "fable_pyculator").exists():
            return candidate
    raise RuntimeError("Could not find the fable-pyculator repository root.")


repo_root = find_repo_root(Path.cwd().resolve())
paths = freshforge_2021_build_paths(
    repo_root=repo_root,
    workflow_filename="freshforge-modelwright-workflow.json",
)
paths.artifact_dir.mkdir(parents=True, exist_ok=True)

workbook_path = paths.workbook_path
artifact_dir = paths.artifact_dir
output_refs_path = paths.output_refs_path
workflow_path = paths.workflow_path
contract_path = paths.contract_path
expressions_path = paths.expressions_path
constants_path = paths.constants_path
inference_result_path = paths.inference_result_path
generation_result_path = paths.generation_result_path
generated_model_path = paths.generated_model_path
generated_values_path = paths.generated_values_path
validation_scenario_path = paths.validation_scenario_path
evaluation_report_path = paths.evaluation_report_path

display(Markdown(f"Workbook path: `{workbook_path.relative_to(repo_root)}`"))
display(Markdown(f"Artifact directory: `{artifact_dir.relative_to(repo_root)}`"))


In [ ]:
missing = []
if not workbook_path.exists():
    missing.append(workbook_path.relative_to(repo_root))

if missing:
    display(Markdown("Missing local artifact(s): " + ", ".join(f"`{path}`" for path in missing)))
    ARTIFACTS_READY = False
else:
    display(Markdown("Ready: 2021 source workbook is available."))
    ARTIFACTS_READY = True

## Derive Modelwright output refs from FABLE workbook structure

Modelwright needs explicit output refs. FABLE Pyculator helps by inspecting FABLE output tables and selecting workbook cells from columns tagged `OUTPUT-*`.

In [ ]:
if ARTIFACTS_READY:
    fable_spec = build_2021_notebook_spec(workbook_path)
    output_refs = derive_output_refs(fable_spec, column_flavour_tags="OUTPUT-*")
    write_output_refs(output_refs_path, output_refs)
    display(Markdown(f"Wrote `{len(output_refs):,}` output refs to `{output_refs_path.relative_to(repo_root)}`."))
    display(output_refs[:10])
else:
    output_refs = []
    display(Markdown("Restore the 2021 workbook before deriving output refs."))


## Prepare a cached-workbook validation scenario

The scenario is generated under `tmp/` from the same selected output refs. Blank cached workbook values are skipped here because they are not comparable as generated-model outputs.

In [ ]:
if ARTIFACTS_READY and output_refs:
    validation_scenario = build_cached_workbook_validation_scenario(
        workbook_path,
        output_refs,
        generated_model_path=generated_model_path,
        scenario_id="fable-c-2021-freshforge-build",
        description="Cached-workbook validation slice derived from FABLE Pyculator OUTPUT-* refs.",
        source_workbook=workbook_path.relative_to(repo_root),
        generated_model=generated_model_path.relative_to(repo_root),
    )
    write_validation_scenario(validation_scenario_path, validation_scenario)
    display(Markdown(f"Wrote `{len(validation_scenario['outputs']):,}` comparable validation outputs to `{validation_scenario_path.relative_to(repo_root)}`."))
else:
    display(Markdown("Restore the 2021 workbook and derive output refs before writing a validation scenario."))


## Declare the FreshForge workflow graph

The graph mirrors the Modelwright stages: infer the generated-model contract, generate Python, execute the generated model, and evaluate it against cached workbook values. The graph is written as JSON because FreshForge accepts YAML or JSON workflow specs.

In [ ]:
workflow_document = build_modelwright_freshforge_workflow(
    paths,
    workdir=repo_root,
    workflow_id="fable_2021_modelwright_build",
    name="FABLE 2021 Modelwright build plan",
    description="Plan-first FreshForge graph for rebuilding the 2021 FABLE generated model.",
    module_name="generated_fable_2021_model",
)
write_freshforge_workflow(workflow_path, workflow_document)
display(Markdown(f"Wrote FreshForge workflow graph to `{workflow_path.relative_to(repo_root)}`."))


## Validate and plan with FreshForge

This proves the workflow graph is structurally coherent and that the Modelwright FreshForge provider is discoverable. It still does not run Modelwright.

In [ ]:
try:
    from freshforge.loading import load_workflow
    from freshforge.planning import create_run_plan

    FRESHFORGE_READY = True
except ModuleNotFoundError:
    FRESHFORGE_READY = False
    display(Markdown("FreshForge is not installed. Install it before planning the graph:"))
    display(Markdown("`python -m pip install \"freshforge @ git+https://github.com/UBC-FRESH/freshforge.git@v0.1.0a1\"`"))

if FRESHFORGE_READY:
    workflow_spec, diagnostics = load_workflow(workflow_path)
    plan = create_run_plan(workflow_spec, diagnostics=diagnostics) if workflow_spec is not None else None
    display(plan.to_dict() if plan is not None else [diagnostic.to_dict() for diagnostic in diagnostics])
    if plan is None or plan.has_errors:
        raise RuntimeError("FreshForge could not plan the Modelwright workflow graph.")

## Build commands

FreshForge has planned the graph. Modelwright does the build. Review these commands first, then set `RUN_BUILD = True` to run them from the notebook.

In [ ]:
commands = [
    [
        sys.executable,
        "-m",
        "modelwright.cli",
        "model",
        "infer-contract",
        str(workbook_path),
        "--module-name",
        "generated_fable_2021_model",
        "--output-refs-file",
        str(output_refs_path),
        "--contract",
        str(contract_path),
        "--expressions",
        str(expressions_path),
        "--constants",
        str(constants_path),
        "--verbose",
    ],
    [
        sys.executable,
        "-m",
        "modelwright.cli",
        "model",
        "generate",
        "--contract",
        str(contract_path),
        "--expressions",
        str(expressions_path),
        "--constants",
        str(constants_path),
        "--out",
        str(generated_model_path),
    ],
    [
        sys.executable,
        "-m",
        "modelwright.cli",
        "model",
        "execute",
        "--contract",
        str(contract_path),
        "--model",
        str(generated_model_path),
    ],
    [
        sys.executable,
        "-m",
        "modelwright.cli",
        "validation",
        "evaluate",
        "--contract",
        str(contract_path),
        "--model",
        str(generated_model_path),
        "--scenario",
        str(validation_scenario_path),
        "--workbook",
        str(workbook_path),
        "--verbose",
    ],
]

for command in commands:
    display(Markdown(f"```bash\n{shlex.join(command)}\n```"))

In [ ]:
RUN_BUILD = False

if RUN_BUILD:
    if not ARTIFACTS_READY:
        raise RuntimeError("Restore the 2021 workbook before running the build.")
    with inference_result_path.open("w", encoding="utf-8") as target:
        subprocess.run(commands[0], check=True, stdout=target)
    with generation_result_path.open("w", encoding="utf-8") as target:
        subprocess.run(commands[1], check=True, stdout=target)
    with generated_values_path.open("w", encoding="utf-8") as target:
        subprocess.run(commands[2], check=True, stdout=target)
    with evaluation_report_path.open("w", encoding="utf-8") as target:
        subprocess.run(commands[3], check=True, stdout=target)
    display(Markdown(f"Generated model: `{generated_model_path.relative_to(repo_root)}`"))
else:
    display(Markdown("Build not run. Set `RUN_BUILD = True` after reviewing the plan and commands."))

## After the build

Once `generated_fable_2021_model.py` exists, the regular 2021 notebook loop can use it at:

- `tmp/generated-models/fable-2021/generated_fable_2021_model.py`

A successful build is not automatically a new equivalence claim. Treat generated outputs as validated only after comparing against the same 2021 source workbook under a documented validation contract.